## Traffic Incident System Dataset

https://opendata.transport.vic.gov.au/dataset/victoria-road-crash-data/resource/5df1f373-0c90-48f5-80e1-7b2a35507134

https://www.police.vic.gov.au/road-policing-statistics

In [0]:
%pip install osmnx folium
%restart_python

In [0]:
dbutils.widgets.text("catalog", "", "Catalog")
catalog = dbutils.widgets.get("catalog")

dbutils.widgets.text("schema", "", "Schema")
schema = dbutils.widgets.get("schema")

dbutils.widgets.text("volume", "", "Volume")
volume = dbutils.widgets.get("volume")

In [0]:
%skip
import requests

url = "https://opendata.transport.vic.gov.au/dataset/bb77800e-1857-4edc-bf9e-e188437a1c8e/resource/5df1f373-0c90-48f5-80e1-7b2a35507134/download/victorian_road_crash_data.csv"
response = requests.get(url)

with open(f'/Volumes/{catalog}/{schema}/{volume}/victorian_road_crash_data.csv', 'wb') as f:
    f.write(response.content)

In [0]:
%skip
(
  spark.read.csv(f'/Volumes/{catalog}/{schema}/{volume}/victorian_road_crash_data.csv', header=True, inferSchema=True)
  .write.mode('overwrite')
  .saveAsTable(f'{catalog}.{schema}.victorian_road_crash_data')
)

In [0]:
%sql
SELECT * FROM ${catalog}.${schema}.victorian_road_crash_data

In [0]:
%sql
DECLARE OR REPLACE VARIABLE victorian_road_crash_data_fqn STRING;
SET VAR victorian_road_crash_data_fqn = concat(:catalog, '.', :schema, '.victorian_road_crash_data');

In [0]:
%sql
SELECT * FROM IDENTIFIER(victorian_road_crash_data_fqn)

In [0]:
%sql
SELECT *, st_astext(ST_Buffer(st_point(longitude, latitude), 0.01)) as buffered_wkt 
FROM IDENTIFIER(victorian_road_crash_data_fqn)
WHERE LGA_NAME = "SHEPPARTON"


In [0]:
# 0.05 buffer is around 5.55 km in radius at the equator
pdf_buffered_accident_spot = spark.sql("""
                                        SELECT *, st_astext(ST_Buffer(st_point(longitude, latitude), 1)) as buffered_wkt 
                                        FROM IDENTIFIER(victorian_road_crash_data_fqn)
                                        WHERE LGA_NAME = "SHEPPARTON"
                                       """).toPandas()

In [0]:
import osmnx
import shapely.geometry
import shapely.wkt

polygon_geom = shapely.wkt.loads(pdf_buffered_accident_spot.iloc[0]['buffered_wkt'])
police_hospital_pdf = osmnx.features.features_from_polygon(polygon_geom, {'amenity': ['police', 'hospital']})

# Cast police_hospital_pdf geometry to string
police_hospital_pdf['geometry'] = police_hospital_pdf['geometry'].apply(lambda geom: geom.wkt)

In [0]:
df_first_accident_record = spark.createDataFrame([pdf_buffered_accident_spot.iloc[0].to_dict()])
from pyspark.sql.functions import expr
df_first_accident_record = df_first_accident_record.withColumn("centroid_accident", expr("st_astext(st_centroid(st_geomfromtext(buffered_wkt)))"))

display(df_first_accident_record)

In [0]:
from pyspark.sql.functions import expr

police_hospital_df = spark.createDataFrame(police_hospital_pdf)
police_hospital_df = police_hospital_df.withColumn("centroid_police_station", expr("st_astext(st_centroid(st_geomfromtext(geometry)))"))
display(police_hospital_df)

In [0]:
import osmnx as ox
from shapely.geometry import LineString, Point

ox.settings.use_cache = True
#ox.settings.cache_folder = "/Volumes/sandbox/danny_schema/dw_volume/osmnx_cache"

def get_route_with_wkt(start_latlng, end_latlng, route_name="Route"):
    """Get route between two points and return WKT data"""
    
    # Download the street network for the area
    G = ox.graph_from_point(start_latlng, dist=80000, network_type="drive") #Limit to 40KM
    
    # Get the nearest network nodes to the start and end points
    orig_node = ox.nearest_nodes(G, start_latlng[1], start_latlng[0])
    dest_node = ox.nearest_nodes(G, end_latlng[1], end_latlng[0])
    
    # Find the shortest path
    route = ox.shortest_path(G, orig_node, dest_node, weight="length")
    
    # Convert to WKT formats
    start_wkt = Point(start_latlng[1], start_latlng[0]).wkt  # longitude, latitude
    end_wkt = Point(end_latlng[1], end_latlng[0]).wkt
    
    # Convert route nodes to LineString WKT
    coords_list = [(G.nodes[node]['x'], G.nodes[node]['y']) for node in route]
    route_linestring = LineString(coords_list)
    route_wkt = route_linestring.wkt
    
    return {
        'route_name': route_name,
        'start_wkt': start_wkt,
        'end_wkt': end_wkt,
        'route_linestring_wkt': route_wkt
    }

In [0]:
# Extract start and end latlng from dataframes
start_latlng = df_first_accident_record.selectExpr("st_x(st_geomfromtext(centroid_accident)) as lat", 
                                                   "st_y(st_geomfromtext(centroid_accident)) as lng").first() #first accident record
end_latlng = police_hospital_df.selectExpr("st_x(st_geomfromtext(centroid_police_station)) as lat", 
                                  "st_y(st_geomfromtext(centroid_police_station)) as lng", "name").first() #first police_hospital record

# Get route data
route_data = get_route_with_wkt((start_latlng.lng, start_latlng.lat), (end_latlng.lng, end_latlng.lat), end_latlng.name)

# Create DataFrame from list of dictionaries
routes_data = [route_data]  # You can add multiple routes here
df_route = spark.createDataFrame(routes_data)

display(df_route)

In [0]:
import folium
from shapely.wkt import loads

# Extract WKT data
start_wkt = df_route.select("start_wkt").first()[0]
end_wkt = df_route.select("end_wkt").first()[0]
route_wkt = df_route.select("route_linestring_wkt").first()[0]
route_name = df_route.select("route_name").first()[0]

# Convert WKT to shapely geometries
start_point = loads(start_wkt)
end_point = loads(end_wkt)
route_line = loads(route_wkt)

# Create a folium map centered around the start point
m = folium.Map(location=[start_point.y, start_point.x], zoom_start=13)

# Add start and end points to the map
folium.Marker([start_point.y, start_point.x], popup='Accident Spot', icon=folium.Icon(color='green')).add_to(m)
folium.Marker([end_point.y, end_point.x], popup=route_name, icon=folium.Icon(color='red')).add_to(m)

# Add the route line to the map
folium.PolyLine([(point[1], point[0]) for point in route_line.coords], color='blue', weight=2.5, opacity=1).add_to(m)

# Display the map
m

%md
# Victorian Road Crash Data: Exploratory Data Analysis (EDA) Summary

This notebook provides an exploratory analysis of the Victorian Road Crash Data, sourced from the Traffic Incident System. The analysis covers schema inspection, column statistics, and visualizations to help understand crash patterns across Victoria.

**Key Findings:**
* The dataset contains detailed records of road crashes, including accident type, severity, date/time, location, and demographic information.
* Most accidents are classified as "Other injury accident" or "Serious injury accident"; fatal accidents are relatively rare.
* The majority of crashes involve a "Collision with vehicle". Other common types include collisions with fixed objects and incidents involving pedestrians.
* Crash counts per year show a dip around 2020, likely due to pandemic-related changes in traffic patterns, but generally remain high.
* Melbourne and surrounding metropolitan LGAs have the highest crash counts, with additional clusters in regional centers.
* Geographic distribution plots show that crashes are densely clustered in urban areas, especially around Melbourne.

**Visualizations included:**
* Bar charts for accident severity, accident type, crashes per year, and top 10 LGAs by crash count.
* Scatter plot of crash locations by latitude and longitude.
* ydata-profiling report for detailed column statistics and missing value analysis.

For further analysis, you can filter by time period, region, or specific accident types, and explore relationships between variables such as severity, location, and time.


%md
## Step 1: Inspect Table Schema and Sample Data
To begin EDA, we need to understand the structure and typical values in the dataset. We'll display the schema and a small sample (20 rows) from the 'danny_catalog.dtp_schema.victorian_road_crash_data' table. This will help us identify key columns and data types for further analysis.

In [0]:
# Display the schema of the table
road_crash_df = spark.table(f'{catalog}.{schema}.victorian_road_crash_data')
road_crash_df.printSchema()

# Show a sample of 20 rows to understand typical values
display(road_crash_df.limit(20))

%md
## Step 2: Generate Column Statistics and Data Profiling
Now that we have inspected the schema and sample data, the next step is to generate a detailed data profile. We'll use ydata-profiling to create a profile report, which will summarize statistics, missing values, and distributions for all columns. We'll sample up to 10,000 rows to avoid excessive memory usage, as the dataset may be large.

In [0]:
# Install ydata-profiling if not already installed
%pip install ydata-profiling==4.8.3

import pandas as pd
from ydata_profiling import ProfileReport

# Sample up to 10,000 rows for profiling to avoid memory issues
profile_sample = road_crash_df.limit(10000).toPandas()

# Fix date/time columns for profiling
if 'ACCIDENT_DATE' in profile_sample.columns:
    profile_sample['ACCIDENT_DATE'] = pd.to_datetime(profile_sample['ACCIDENT_DATE'], errors='coerce')
if 'ACCIDENT_TIME' in profile_sample.columns:
    profile_sample['ACCIDENT_TIME'] = profile_sample['ACCIDENT_TIME'].astype(str)

# Generate the profile report
profile = ProfileReport(profile_sample, title="Victorian Road Crash Data Profiling Report", explorative=True)

# Display the report in notebook (HTML)
profile.to_widgets()

%md
## Step 3: Visualize Distributions of Key Columns
Now that we have a data profile, let's visualize the distributions of important columns. We'll focus on SEVERITY, ACCIDENT_TYPE, ACCIDENT_DATE, and LGA_NAME, as these are critical for understanding crash patterns. We'll use matplotlib for static plots. For ACCIDENT_DATE, we'll plot the number of crashes per year. For SEVERITY and ACCIDENT_TYPE, we'll show bar charts of their frequencies. For LGA_NAME, we'll show the top 10 LGAs by crash count. If latitude/longitude is present, we'll plot a scatter map of crash locations.

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Sample up to 10,000 rows for visualization
viz_sample = road_crash_df.limit(10000).toPandas()

fig, axs = plt.subplots(2, 2, figsize=(16, 12))

# SEVERITY distribution
viz_sample['SEVERITY'].value_counts().plot(kind='bar', ax=axs[0,0], color='skyblue')
axs[0,0].set_title('Accident Severity Distribution')
axs[0,0].set_xlabel('Severity')
axs[0,0].set_ylabel('Count')

# ACCIDENT_TYPE distribution
viz_sample['ACCIDENT_TYPE'].value_counts().plot(kind='bar', ax=axs[0,1], color='salmon')
axs[0,1].set_title('Accident Type Distribution')
axs[0,1].set_xlabel('Accident Type')
axs[0,1].set_ylabel('Count')

# ACCIDENT_DATE: crashes per year
if 'ACCIDENT_DATE' in viz_sample.columns:
    viz_sample['ACCIDENT_YEAR'] = pd.to_datetime(viz_sample['ACCIDENT_DATE'], errors='coerce').dt.year
    viz_sample['ACCIDENT_YEAR'].value_counts().sort_index().plot(kind='bar', ax=axs[1,0], color='lightgreen')
    axs[1,0].set_title('Crashes per Year')
    axs[1,0].set_xlabel('Year')
    axs[1,0].set_ylabel('Count')

# LGA_NAME: top 10 LGAs by crash count
viz_sample['LGA_NAME'].value_counts().head(10).plot(kind='bar', ax=axs[1,1], color='orchid')
axs[1,1].set_title('Top 10 LGAs by Crash Count')
axs[1,1].set_xlabel('LGA Name')
axs[1,1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [0]:
# Plot latitude/longitude scatter if present
if 'LATITUDE' in viz_sample.columns and 'LONGITUDE' in viz_sample.columns:
    plt.figure(figsize=(10,8))
    plt.scatter(viz_sample['LONGITUDE'], viz_sample['LATITUDE'], alpha=0.3, s=10, c='blue')
    plt.title('Geographic Distribution of Road Crashes')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.show()